# Telecom Decision Model Benchmark: Jev vs Laya vs OTel-2.0 vs Claude Sonnet vs Gemini

A controlled experiment: the **same telecom incidents** and the **same policy text** go to five decision-makers, and we measure
**quality, latency, throughput, scalability, reliability and cost**.

| System | What it is | How we call it |
|---|---|---|
| **Jev** (`jev-1.13`) | TypeSafe AI "System One" decision model; returns typed decisions + probabilities | HTTP API (OpenRouter alpha endpoint, or `https://api.typesafe.ai/v1/systemone`) |
| **Laya** | Small (~421M) typed-decision encoder, same idea as Jev (`choice` / `noul` / `score`, calibrated confidence) | **Runs locally**: `from laya import Router` |
| **OTel-2.0-Decision** (`att/OTel-2.0-Decision-26B-A4B-4bit`) | AT&T's 4-bit Gemma-4 26B-A4B typed-decision model; the answer is read from **label log-probabilities**, not generated text | OpenAI-compatible `/v1/chat/completions` with `logprobs` (AT&T test endpoint, your own vLLM server, or OpenRouter if it is ever listed) |
| **Claude Sonnet** | Frontier generative LLM | OpenRouter chat completions |
| **Gemini** | Frontier generative LLM | OpenRouter chat completions |

**Notebook map:** 1 Install → 2 Keys & config → 3 Simulator → 4 Backends (4b = OTel) → 5 Engine → 6 Smoke test → 7 Full run → 8 Analysis.

> **OTel note.** OTel needs one request per question (4 per incident). Its latency is the 4 calls run in parallel. The AT&T test endpoint is shared, unauthenticated and *logged*, so only synthetic data goes there.
>
> **Fairness note.** Jev, Sonnet and Gemini are remote APIs, so their latency includes the network. Laya runs on *your* hardware
> (Colab GPU/CPU), so its latency is compute only, and its cost is GPU-time (`LAYA_HOURLY_USD`), not tokens. Jev's speed/price claims are
> company-reported; this notebook produces *your* numbers.

In [ ]:
import os

os.environ["LAYA_MAX_LEN"] = "1800"
os.environ["LAYA_MODEL"]   = "typed-decisions"   # the 421M checkpoint with the 1024 context

## 1. Install

In [ ]:
# %pip install -q httpx python-dotenv matplotlib pandas seaborn
# Laya runs locally and pulls in torch + transformers (Python 3.10+). First use also downloads a model checkpoint.
%pip install -q laya

## 2. API keys & configuration
One **OpenRouter** key covers Jev, Sonnet and Gemini. Laya needs **no key** (it runs locally). OTel needs **no key** on AT&T's test endpoint.
The key is read from the environment / `.env`, or you're prompted securely. **Never paste keys into chat, notebook cells or comments, and never commit them.**

- `JEV_PROVIDER`: `openrouter` (key: `OPENROUTER_API_KEY`) or `typesafe` (key: `TYPESAFE_API_KEY`)
- Laya knobs (all optional): `LAYA_MODEL` (e.g. `typed-decisions`), `LAYA_MAX_LEN` (e.g. `1024`; the policy text is long, so raise it if you suspect truncation), `LAYA_SERIALIZE`, `LAYA_HOURLY_USD`
- Prices: Jev defaults to $0.042 / 1M input tokens. Sonnet and Gemini prices are pulled live from OpenRouter's catalog.

In [ ]:
import os, getpass
try:
    from dotenv import load_dotenv, find_dotenv; load_dotenv(find_dotenv(usecwd=True))
except ImportError:
    pass

# ---- which systems to benchmark ---------------------------------------------
BACKENDS   = ["jev", "laya", "otel"]   # add "sonnet", "gemini" if you want them too

# ---- Jev ----------------------------------------------------------------------
os.environ["JEV_PROVIDER"] = "openrouter"            # "openrouter" | "typesafe"
JEV_MODEL  = "jev-1.13"                              # or "jev-latest"

# ---- frontier LLMs on OpenRouter (check slugs at openrouter.ai/models) -------
SONNET_SLUG = "anthropic/claude-sonnet-4.6"
GEMINI_SLUG = "google/gemini-2.5-flash"

# ---- OTel-2.0-Decision (AT&T) ---------------------------------------------------
# Default = AT&T's public TEST endpoint (no key, shared, logged, no uptime guarantee -> synthetic data only).
# Other options:   your own vLLM server -> OTEL_BASE_URL="http://localhost:8000/v1"
#                  OpenRouter (only if the model is listed there, see the check cell in section 4b)
#                    -> OTEL_BASE_URL="https://openrouter.ai/api/v1" and OTEL_MODEL="att/OTel-2.0-Decision-26B-A4B-4bit"
os.environ.setdefault("OTEL_BASE_URL", "https://otel-2-decision-26b-a4b-4bit.cdoresearch.ai/v1")
os.environ.setdefault("OTEL_MODEL", "OTel-2.0-Decision-26B-A4B-4bit")
os.environ.setdefault("OTEL_MIN_MASS", "0.5")          # answer_mass below this = invalid readout
os.environ.setdefault("OTEL_PRICE_IN_PER_M", "0")      # $ / 1M input tokens (0 = free test endpoint)
# os.environ["OTEL_API_KEY"] = "..."                   # only if your server needs one

# ---- Laya (local) ---------------------------------------------------------------
os.environ.setdefault("LAYA_SERIALIZE", "1")         # 1 = one model instance, requests queue (realistic single-GPU serving)
os.environ.setdefault("LAYA_HOURLY_USD", "0.35")     # GPU price used to turn Laya compute-seconds into $ (T4 ~ $0.35/h). 0 = free Colab
# os.environ["LAYA_MODEL"]   = "typed-decisions"     # fine-tuned checkpoint (default: router's choice)
# os.environ["LAYA_MAX_LEN"] = "1024"                # context budget for state + questions

# ---- keys (prompted only if missing) ------------------------------------------
need = set()
if "jev" in BACKENDS:
    need.add("OPENROUTER_API_KEY" if os.environ["JEV_PROVIDER"] == "openrouter" else "TYPESAFE_API_KEY")
if "sonnet" in BACKENDS or "gemini" in BACKENDS:
    need.add("OPENROUTER_API_KEY")
if "otel" in BACKENDS and "openrouter.ai" in os.environ["OTEL_BASE_URL"]:
    need.add("OPENROUTER_API_KEY")
for k in need:
    if not os.environ.get(k):
        os.environ[k] = getpass.getpass(f"{k}: ")

## 3. Simulator: shared policy, incident generator, ground truth
Both systems receive the *identical* policy text below, so the only variable is the model.

In [ ]:
# --------------------------------------------------------------------------- #
# 1. Shared policy (used verbatim by every system)
# --------------------------------------------------------------------------- #
# Field names below match the labels in the incident text ("Users affected", "Packet loss",
# "Latency", "Outage duration", "Status", "Suspected cause", element type). Causes are written
# exactly as they appear in that text ("power failure", not power_failure).

# Severity is described once and reused, so every question uses identical wording for it.
_SEVERITY_ORDER = "Check the levels in this order and stop at the first one that matches: CRITICAL, HIGH, MEDIUM, LOW."
_SEVERITY_DEFS = (
    "CRITICAL = status is DOWN AND (users affected >= 5000 OR element type is CORE_COMPONENT). "
    "HIGH = not CRITICAL AND (users affected >= 1000 OR (packet loss >= 30% AND users affected >= 300)). "
    "MEDIUM = not CRITICAL and not HIGH AND (users affected >= 100 OR packet loss >= 10% OR latency >= 250 ms). "
    "LOW = none of the above (users affected < 100, packet loss < 10%, latency < 250 ms, "
    "and not a DOWN CORE_COMPONENT)."
)

# ---- Q1: severity ---------------------------------------------------------- #
SEVERITY_CRITERIA = {
    "LOW": "None of the levels below apply: users affected < 100, packet loss < 10%, latency < 250 ms, "
           "and not a DOWN CORE_COMPONENT.",
    "MEDIUM": "Not CRITICAL and not HIGH, AND (users affected >= 100 OR packet loss >= 10% OR latency >= 250 ms).",
    "HIGH": "Not CRITICAL, AND (users affected >= 1000 OR (packet loss >= 30% AND users affected >= 300)).",
    "CRITICAL": "Status is DOWN AND (users affected >= 5000 OR element type is CORE_COMPONENT).",
}
SEVERITY_INSTRUCTIONS = (
    "Classify the severity of this telecom incident. "
    + _SEVERITY_ORDER
    + " If an incident meets the conditions of more than one level, choose the highest one."
)

# ---- Q2: responsible team -------------------------------------------------- #
RESPONSIBLE_CRITERIA = {
    "RAN": "The element type is 5G_BASE_STATION and the cause is neither power failure nor fiber cut.",
    "CORE": "The element type is GATEWAY or CORE_COMPONENT and the cause is neither power failure nor fiber cut.",
    "TRANSPORT": "The suspected cause is fiber cut (any element type), "
                 "OR the element type is BACKHAUL_LINK and the cause is not power failure.",
    "IP_ROUTING": "The element type is ROUTER and the cause is neither power failure nor fiber cut.",
    "FACILITIES": "The suspected cause is power failure (any element type).",
}
RESPONSIBLE_INSTRUCTIONS = (
    "Which team is responsible for resolving this incident? "
    "The suspected cause decides first: power failure always goes to FACILITIES and fiber cut always goes to "
    "TRANSPORT, whatever the element type. For every other cause, decide by element type: "
    "5G_BASE_STATION -> RAN, GATEWAY or CORE_COMPONENT -> CORE, ROUTER -> IP_ROUTING, BACKHAUL_LINK -> TRANSPORT."
)

# ---- Q3: escalate (yes / no) ------------------------------------------------ #
ESCALATE_INSTRUCTIONS = (
    "Should this incident be escalated? Answer YES if the severity is HIGH or CRITICAL, or if the severity is "
    "MEDIUM and the outage duration is 30 minutes or more. Answer NO if the severity is LOW, or if it is MEDIUM "
    "and the outage duration is under 30 minutes. Duration does not matter for HIGH or CRITICAL. "
    "To get the severity: " + _SEVERITY_ORDER + " " + _SEVERITY_DEFS
)
ESCALATE_TRUE = "Severity is HIGH or CRITICAL, or severity is MEDIUM and outage duration is 30 minutes or more."
ESCALATE_FALSE = "Severity is LOW, or severity is MEDIUM and outage duration is under 30 minutes."

# ---- Q4: remediation action ------------------------------------------------- #
ACTION_CRITERIA = {
    "MONITOR": "Severity is LOW. This applies whatever the suspected cause is.",
    "REMOTE_RESTART": "Severity is not LOW AND the suspected cause is software crash.",
    "REROUTE_TRAFFIC": "Severity is not LOW AND the suspected cause is congestion.",
    "ROLLBACK_CONFIG": "Severity is not LOW AND the suspected cause is config error.",
    "SCHEDULE_MAINTENANCE": "Severity is MEDIUM AND the suspected cause is power failure, fiber cut or hardware fault.",
    "DISPATCH_ENGINEER": "Severity is HIGH or CRITICAL AND the suspected cause is power failure, fiber cut "
                         "or hardware fault.",
}
ACTION_INSTRUCTIONS = (
    "Choose exactly one remediation action. First work out the severity. " + _SEVERITY_ORDER + " " + _SEVERITY_DEFS
    + " Then: if the severity is LOW, choose MONITOR whatever the cause. Otherwise choose by suspected cause: "
    "software crash -> REMOTE_RESTART; congestion -> REROUTE_TRAFFIC; config error -> ROLLBACK_CONFIG; "
    "power failure, fiber cut or hardware fault -> SCHEDULE_MAINTENANCE if the severity is MEDIUM, "
    "or DISPATCH_ENGINEER if the severity is HIGH or CRITICAL."
)

In [ ]:
# --------------------------------------------------------------------------- #
# 2. Incident model, generator and rule-based ground truth
#    (field labels in to_text() match the policy wording above)
# --------------------------------------------------------------------------- #
import random
from dataclasses import dataclass

ELEMENT_TYPES = ["5G_BASE_STATION", "GATEWAY", "CORE_COMPONENT", "ROUTER", "BACKHAUL_LINK"]
STATUSES = ["DOWN", "DEGRADED"]
CAUSES = ["software_crash", "congestion", "config_error", "power_failure", "fiber_cut", "hardware_fault"]
REGIONS = ["Mumbai", "Delhi", "Chennai", "Bengaluru", "Kolkata", "Hyderabad", "Pune", "Ahmedabad"]


@dataclass
class Incident:
    incident_id: int
    element_type: str
    element_id: str
    region: str
    status: str
    users_affected: int
    packet_loss_pct: float
    latency_ms: int
    duration_min: int
    cause: str

    def to_text(self) -> str:
        return (f"Incident {self.incident_id}. Element type: {self.element_type}. Element ID: {self.element_id}. "
                f"Region: {self.region}. Status: {self.status}. Users affected: {self.users_affected}. "
                f"Packet loss: {self.packet_loss_pct}%. Latency: {self.latency_ms} ms. "
                f"Outage duration: {self.duration_min} minutes. "
                f"Suspected cause: {self.cause.replace('_', ' ')}.")


def label_severity(inc: Incident) -> str:
    u, loss, lat = inc.users_affected, inc.packet_loss_pct, inc.latency_ms
    if inc.status == "DOWN" and (u >= 5000 or inc.element_type == "CORE_COMPONENT"):
        return "CRITICAL"
    if u >= 1000 or (loss >= 30 and u >= 300):
        return "HIGH"
    if u >= 100 or loss >= 10 or lat >= 250:
        return "MEDIUM"
    return "LOW"


def label(inc: Incident) -> dict:
    """Rule-based ground truth. It deliberately ignores any narrative notes."""
    sev = label_severity(inc)
    cause = inc.cause.replace(" ", "_").lower()
    # responsible team: cause first, then element type
    if cause == "power_failure":
        team = "FACILITIES"
    elif cause == "fiber_cut":
        team = "TRANSPORT"
    else:
        team = {"5G_BASE_STATION": "RAN", "GATEWAY": "CORE", "CORE_COMPONENT": "CORE",
                "ROUTER": "IP_ROUTING", "BACKHAUL_LINK": "TRANSPORT"}[inc.element_type]
    # escalate
    escalate = "YES" if (sev in ("HIGH", "CRITICAL") or (sev == "MEDIUM" and inc.duration_min >= 30)) else "NO"
    # action
    if sev == "LOW":
        action = "MONITOR"
    elif cause == "software_crash":
        action = "REMOTE_RESTART"
    elif cause == "congestion":
        action = "REROUTE_TRAFFIC"
    elif cause == "config_error":
        action = "ROLLBACK_CONFIG"
    else:  # power failure, fiber cut, hardware fault
        action = "SCHEDULE_MAINTENANCE" if sev == "MEDIUM" else "DISPATCH_ENGINEER"
    return {"severity": sev, "responsible": team, "escalate": escalate, "action": action}


def generate_incidents(n: int, seed: int = 42, hard_fraction: float = 0.3) -> list:
    """n seeded incidents. A `hard_fraction` share has metrics sitting right on a policy threshold."""
    rng = random.Random(seed)
    users_edges = [99, 100, 101, 299, 300, 301, 999, 1000, 1001, 4999, 5000, 5001]
    loss_edges = [9.9, 10.0, 10.1, 29.9, 30.0, 30.1]
    lat_edges = [249, 250, 251]
    dur_edges = [29, 30, 31]
    out = []
    for i in range(n):
        hard = rng.random() < hard_fraction
        if hard:
            users = rng.choice(users_edges) if rng.random() < 0.7 else int(10 ** rng.uniform(0, 3.8))
            loss = rng.choice(loss_edges) if rng.random() < 0.6 else round(rng.uniform(0, 45), 1)
            lat = rng.choice(lat_edges) if rng.random() < 0.6 else rng.randint(20, 600)
            dur = rng.choice(dur_edges) if rng.random() < 0.6 else rng.randint(1, 180)
        else:
            users = int(10 ** rng.uniform(0, 4.3))
            loss = round(rng.uniform(0, 60), 1)
            lat = rng.randint(20, 700)
            dur = rng.randint(1, 240)
        et = rng.choice(ELEMENT_TYPES)
        out.append(Incident(
            incident_id=i + 1, element_type=et, element_id=f"{et.split('_')[0]}-{rng.randint(100, 999)}",
            region=rng.choice(REGIONS), status=rng.choice(STATUSES), users_affected=users,
            packet_loss_pct=loss, latency_ms=lat, duration_min=dur, cause=rng.choice(CAUSES)))
    return out


# quick sanity check of the ground-truth rules
_demo = generate_incidents(3, seed=1)
for _d in _demo:
    print(_d.to_text(), "->", label(_d))

In [ ]:
import random
from dataclasses import dataclass

@dataclass
class ComplexIncident(Incident):
    # Adding a narrative field containing unstructured telemetry, jargon, and conflicting contextual clues
    operator_notes: str = ""

    def to_text(self) -> str:
        base_text = super().to_text()
        return f"{base_text} Operator Notes & Logs: \"{self.operator_notes}\"."

# Let's generate data samples specifically designed to test the boundary limits of Jev's structured parser
# and highlight the unstructured reasoning strengths of LLMs.
COMPLEX_SCENARIOS = [
    {
        "element_type": "5G_BASE_STATION",
        "status": "DEGRADED",
        "users_affected": 95, # Barely under LOW threshold
        "packet_loss_pct": 8.5,
        "latency_ms": 240,
        "duration_min": 45,
        "cause": "software_crash",
        "operator_notes": "Engineers on-site noted that although official dashboards show 95 active user sessions, a local festival has gathered 2000+ unregistered standby users near the cell tower waiting for reconnection. Major congestion is imminent."
        # An LLM can reason about the '2000+ standby users' and escalate or bump severity, whereas strict schemas look only at the 95 users parameter.
    },
    {
        "element_type": "ROUTER",
        "status": "DOWN",
        "users_affected": 150,
        "packet_loss_pct": 15.0,
        "latency_ms": 260,
        "duration_min": 15,
        "cause": "config_error",
        "operator_notes": "An automated deployment script pushed a faulty routing table. However, backup fiber routes are carrying the load successfully, reducing actual performance impact to minimal levels. Ignore raw metrics."
        # LLMs can identify the negation ('backup routes carrying load', 'ignore raw metrics') and adjust action/severity accordingly.
    },
    {
        "element_type": "CORE_COMPONENT",
        "status": "DOWN",
        "users_affected": 6000,
        "packet_loss_pct": 95.0,
        "latency_ms": 1000,
        "duration_min": 120,
        "cause": "power_failure",
        "operator_notes": "CRITICAL ALERT: Primary generator tripped. Backup batteries are failing. However, the system is scheduled for decommissioning in exactly 5 minutes, and all traffic has already been fully migrated to CORE-02."
        # Rule-based models will classify this as CRITICAL with an urgent DISPATCH_ENGINEER action, but a human or an advanced LLM would realize the decommissioning makes this a low priority MONITOR action.
    }
]

def generate_complex_incidents() -> list[ComplexIncident]:
    incidents = []
    for i, sc in enumerate(COMPLEX_SCENARIOS):
        inc = ComplexIncident(
            incident_id=1000 + i,
            element_type=sc["element_type"],
            element_id=f"C-{1000+i}",
            region="Mumbai",
            status=sc["status"],
            users_affected=sc["users_affected"],
            packet_loss_pct=sc["packet_loss_pct"],
            latency_ms=sc["latency_ms"],
            duration_min=sc["duration_min"],
            cause=sc["cause"],
            operator_notes=sc["operator_notes"]
        )
        incidents.append(inc)
    return incidents

# Preview the complex unstructured samples
for ci in generate_complex_incidents():
    print(f"--- Incident {ci.incident_id} ---")
    print(ci.to_text())
    print()

In [ ]:
def generate_combined_benchmark_dataset(n_standard: int, seed: int = 42, hard_fraction: float = 0.3) -> list[Incident]:
    # 1. Generate standard incidents
    dataset = generate_incidents(n_standard, seed=seed, hard_fraction=hard_fraction)

    # 2. Append the complex incidents we created to challenge Jev
    complex_dataset = generate_complex_incidents()
    dataset.extend(complex_dataset)

    print(f"Combined dataset prepared: {n_standard} standard incidents + {len(complex_dataset)} edge-case complex incidents.")
    return dataset

# Let's verify the dataset size and look at the last element (which should be one of our complex edge cases)
test_dataset = generate_combined_benchmark_dataset(50)
print("Last incident in combined dataset:", test_dataset[-1].to_text())

## 4. Backends: Jev, Laya, OTel, Sonnet, Gemini
Every backend exposes `async decide(client, incident) -> CallResult`, so the runner treats them identically. Failed or invalid outputs are recorded, never silently dropped.
Jev, Laya and OTel all return typed answers (`choice` + confidence, `noul` = P(yes)); Sonnet and Gemini must produce JSON, which is validated against the same allowed labels.

In [ ]:
"""
backends.py
-----------
Every backend exposes:   async decide(client, incident) -> CallResult
so the benchmark runner treats all five systems identically.

  JevBackend          -> TypeSafe System One API   (name "jev")
  LayaBackend         -> Laya Router, runs LOCALLY (name "laya")      pip install laya
  OTelBackend         -> AT&T OTel-2.0-Decision via logprobs (name "otel"), defined in section 4b
  OpenRouterBackend   -> Claude Sonnet / Gemini through OpenRouter    (names "sonnet", "gemini")
"""

import asyncio
import json
import os
import time
from dataclasses import dataclass, field

import httpx


FIELDS = ("severity", "responsible", "escalate", "action")
ALLOWED = {
    "severity": list(SEVERITY_CRITERIA),
    "responsible": list(RESPONSIBLE_CRITERIA),
    "escalate": ["YES", "NO"],
    "action": list(ACTION_CRITERIA),
}


@dataclass
class CallResult:
    decision: dict | None = None        # validated {"severity":..., "responsible":..., "escalate":..., "action":...}
    latency_s: float = 0.0              # wall time the caller waited, INCLUDING retries / queueing
    compute_s: float = 0.0              # Laya only: pure model time (excludes waiting for the lock)
    transport_ok: bool = False          # call completed (HTTP 200 for APIs, no exception for local Laya)
    valid: bool = False                 # output parsed AND every field is an allowed label
    strict_json: bool | None = None     # LLM: was the raw reply pure JSON?  Jev/Laya: typed by construction
    error: str | None = None
    input_tokens: int = 0
    output_tokens: int = 0
    retries: int = 0
    confidence: dict = field(default_factory=dict)   # Jev / Laya: per-field confidence
    raw: str = ""


def validate(decision) -> dict | None:
    """Normalise and check a decision dict. Returns None if anything is missing/illegal."""
    if not isinstance(decision, dict):
        return None
    out = {}
    for k in FIELDS:
        v = decision.get(k)
        if isinstance(v, bool):
            v = "YES" if v else "NO"
        if not isinstance(v, str):
            return None
        v = v.strip().upper().replace(" ", "_")
        if v not in ALLOWED[k]:
            return None
        out[k] = v
    return out


def validate_partial(decision) -> dict:
    """Like validate(), but keeps the good fields and sets bad/missing ones to None."""
    out = {}
    for k in FIELDS:
        v = decision.get(k) if isinstance(decision, dict) else None
        if isinstance(v, bool):
            v = "YES" if v else "NO"
        v = v.strip().upper().replace(" ", "_") if isinstance(v, str) else None
        out[k] = v if v in ALLOWED[k] else None
    return out


async def post_with_retry(client: httpx.AsyncClient, url: str, headers: dict, payload: dict, max_retries: int = 2):
    """POST with exponential back-off on 429/5xx/timeouts. Returns (response|None, retries, elapsed, error|None)."""
    retries, t0 = 0, time.perf_counter()
    while True:
        try:
            r = await client.post(url, headers=headers, json=payload)
            if r.status_code in (429, 500, 502, 503, 504) and retries < max_retries:
                retries += 1
                await asyncio.sleep(min(0.5 * 2 ** retries, 4.0))
                continue
            return r, retries, time.perf_counter() - t0, None
        except (httpx.TimeoutException, httpx.TransportError) as e:
            if retries < max_retries:
                retries += 1
                await asyncio.sleep(min(0.5 * 2 ** retries, 4.0))
                continue
            return None, retries, time.perf_counter() - t0, f"{type(e).__name__}: {e}"


# --------------------------------------------------------------------------- #
# shared helpers for typed-decision answers (Jev and Laya use the same answer shape)
# --------------------------------------------------------------------------- #
def _parse_noul(ans: dict):
    """Return (bool|None, confidence|None) for a Noul (yes/no probability) answer.
    {"type": "noul", "noul": 0.04} means P(yes) = 0.04. Other plausible shapes are handled defensively."""
    if not isinstance(ans, dict):
        return None, None
    for key in ("noul", "probability"):
        v = ans.get(key)
        if isinstance(v, (int, float)) and not isinstance(v, bool):
            return v >= 0.5, max(v, 1 - v)
    for key in ("value", "answer", "result", "decision"):
        v = ans.get(key)
        if isinstance(v, bool):
            p = ans.get("probability")
            return v, (max(p, 1 - p) if isinstance(p, (int, float)) else ans.get("confidence"))
    probs = ans.get("probabilities") or {}
    for t, f in (("true", "false"), ("yes", "no")):
        if t in probs:
            pt, pf = probs[t], probs.get(f, 1 - probs[t])
            return pt >= pf, max(pt, pf)
    return None, None


def _choice_conf(a: dict):
    """Confidence of a choice answer: Jev says 'confidence', Laya says 'answer_confidence'."""
    for key in ("answer_confidence", "confidence"):
        v = a.get(key)
        if isinstance(v, (int, float)) and not isinstance(v, bool):
            return float(v)
    probs = a.get("probabilities") or {}
    return max(probs.values()) if probs else None


def _extract_decision(answers: dict):
    """answers -> (decision dict, confidence dict). Raises KeyError if a question is missing."""
    decision, conf = {}, {}
    for k in ("severity", "responsible", "action"):
        a = answers[k]
        decision[k] = a["choice"]
        conf[k] = _choice_conf(a)
    yes, c = _parse_noul(answers["escalate"])
    decision["escalate"] = None if yes is None else ("YES" if yes else "NO")
    conf["escalate"] = c
    return decision, conf


# =========================================================================== #
# Jev (TypeSafe System One)
# =========================================================================== #
class JevBackend:
    """provider = env JEV_PROVIDER: "typesafe" (direct, TYPESAFE_API_KEY) or "openrouter" (OPENROUTER_API_KEY).
    Request/response body is the same System One format on both."""

    def __init__(self, model: str = "jev-1.13", name: str = "jev"):
        self.name = name
        self.provider = os.getenv("JEV_PROVIDER", "typesafe").lower()
        if self.provider == "openrouter":
            self.model = {"jev-1.13": "typesafe/jev-1.13", "jev-latest": "~typesafe/jev-latest"}.get(model, model)
            self.url = os.getenv("JEV_API_URL", "https://openrouter.ai/api/alpha/decisions")
            self.key = os.environ.get("OPENROUTER_API_KEY")
            key_name = "OPENROUTER_API_KEY"
        else:
            self.model = model
            self.url = os.getenv("JEV_API_URL", "https://api.typesafe.ai/v1/systemone")
            self.key = os.environ.get("TYPESAFE_API_KEY")
            key_name = "TYPESAFE_API_KEY"
        if not self.key:
            raise RuntimeError(f"{key_name} is not set (JEV_PROVIDER={self.provider})")

    def build_payload(self, inc: Incident) -> dict:
        return {
            "model": self.model,
            "state": inc.to_text(),
            "questions": {
                "severity": {"type": "choice", "instructions": SEVERITY_INSTRUCTIONS,
                             "criteria": SEVERITY_CRITERIA},
                "responsible": {"type": "choice", "instructions": RESPONSIBLE_INSTRUCTIONS,
                                "criteria": RESPONSIBLE_CRITERIA},
                "escalate": {"type": "noul", "instructions": ESCALATE_INSTRUCTIONS,
                             "criteria": {"true": ESCALATE_TRUE, "false": ESCALATE_FALSE}},
                "action": {"type": "choice", "instructions": ACTION_INSTRUCTIONS,
                           "criteria": ACTION_CRITERIA},
            },
        }

    async def decide(self, client: httpx.AsyncClient, inc: Incident) -> CallResult:
        payload = self.build_payload(inc)
        headers = {"Authorization": f"Bearer {self.key}", "Content-Type": "application/json"}
        r, retries, lat, err = await post_with_retry(client, self.url, headers, payload)
        res = CallResult(latency_s=lat, retries=retries)
        if r is None:
            res.error = err
            return res
        res.raw = r.text[:600]
        if r.status_code != 200:
            res.error = f"HTTP {r.status_code}: {r.text[:200]}"
            return res
        res.transport_ok = True
        try:
            data = r.json()
            answers = data.get("answers") or data.get("results") or data
            decision, conf = _extract_decision(answers)
            res.confidence = conf
            res.decision = validate_partial(decision)
            bad = [k for k, v in res.decision.items() if v is None]
            res.valid = not bad
            if bad:
                res.error = f"unparsed fields: {bad}"
            res.strict_json = True            # typed by construction
            usage = data.get("usage") or {}
            res.input_tokens = usage.get("input_tokens") or usage.get("prompt_tokens") or len(json.dumps(payload)) // 4
            res.output_tokens = usage.get("output_tokens") or usage.get("completion_tokens") or 0
        except Exception as e:                # malformed / unexpected response shape
            res.error = f"parse error: {type(e).__name__}: {e}"
        return res


# =========================================================================== #
# Laya (local Router: pip install laya)
# =========================================================================== #
class LayaBackend:
    """Laya is NOT an HTTP API: `Router.predict(state, questions)` runs a ~421M-parameter encoder on YOUR machine.
    Consequences for the benchmark:
      * latency  = your hardware (GPU/CPU), not a network round-trip;
      * cost     = compute time x hourly price (LAYA_HOURLY_USD), not tokens;
      * requests are serialised with a lock by default (one model instance), so at concurrency>1 the
        measured latency includes queueing -- exactly what production looks like without a batching server.
    Env knobs: LAYA_MODEL (e.g. "typed-decisions"), LAYA_MAX_LEN (e.g. 1024), LAYA_ROUTER_KWARGS (JSON),
               LAYA_SERIALIZE (1/0), LAYA_HOURLY_USD (default 0.35, a T4 GPU hour)."""

    def __init__(self, name: str = "laya"):
        try:
            from laya import Router
        except ImportError as e:
            raise RuntimeError("laya is not installed -> run:  %pip install laya   (Python 3.10+)") from e
        self.name = name
        self.model = os.getenv("LAYA_MODEL") or None                      # None = let the Router choose
        self.max_len = int(os.getenv("LAYA_MAX_LEN", "0")) or None
        self.router = Router(**json.loads(os.getenv("LAYA_ROUTER_KWARGS", "{}")))
        self.lock = asyncio.Lock() if os.getenv("LAYA_SERIALIZE", "1") == "1" else None

    @staticmethod
    def build_questions() -> dict:
        # Same policy text as Jev. Laya's noul type takes instructions only, so the true/false meanings are
        # folded into the instruction string instead of a separate "criteria" block.
        escalate = (f"{ESCALATE_INSTRUCTIONS} Answer YES (true) when: {ESCALATE_TRUE} "
                    f"Answer NO (false) when: {ESCALATE_FALSE}")
        return {
            "severity": {"type": "choice", "instructions": SEVERITY_INSTRUCTIONS, "criteria": SEVERITY_CRITERIA},
            "responsible": {"type": "choice", "instructions": RESPONSIBLE_INSTRUCTIONS,
                            "criteria": RESPONSIBLE_CRITERIA},
            "escalate": {"type": "noul", "instructions": escalate},
            "action": {"type": "choice", "instructions": ACTION_INSTRUCTIONS, "criteria": ACTION_CRITERIA},
        }

    def _predict(self, text: str, questions: dict):
        kw = {}
        if self.model:
            kw["model"] = self.model
        if self.max_len:
            kw["max_len"] = self.max_len
        t0 = time.perf_counter()
        out = self.router.predict(text, questions, **kw)
        return out, time.perf_counter() - t0

    async def decide(self, client, inc: Incident) -> CallResult:
        text, questions = inc.to_text(), self.build_questions()
        res = CallResult()
        t0 = time.perf_counter()
        try:
            if self.lock:
                async with self.lock:
                    out, res.compute_s = await asyncio.to_thread(self._predict, text, questions)
            else:
                out, res.compute_s = await asyncio.to_thread(self._predict, text, questions)
        except Exception as e:
            res.latency_s = time.perf_counter() - t0
            res.error = f"{type(e).__name__}: {e}"
            return res
        res.latency_s = time.perf_counter() - t0
        res.transport_ok = True
        try:
            res.raw = json.dumps(out, default=str)[:600]
            decision, conf = _extract_decision(out["answers"])
            res.confidence = conf
            res.decision = validate_partial(decision)
            bad = [k for k, v in res.decision.items() if v is None]
            res.valid = not bad
            if bad:
                res.error = f"unparsed fields: {bad}"
            res.strict_json = True            # typed by construction
            res.input_tokens = (len(text) + len(json.dumps(questions))) // 4     # estimate (no tokens are billed)
            res.output_tokens = 0
        except Exception as e:
            res.error = f"parse error: {type(e).__name__}: {e}"
        return res


# =========================================================================== #
# LLM prompt (same policy text as Jev / Laya get)
# =========================================================================== #
def _fmt(d: dict) -> str:
    return "\n".join(f"  - {k}: {v}" for k, v in d.items())


def build_llm_messages(inc: Incident) -> list[dict]:
    system = (
        "You are a telecom network-operations triage system. Apply the policy below exactly.\n\n"
        f"SEVERITY ({SEVERITY_INSTRUCTIONS})\n{_fmt(SEVERITY_CRITERIA)}\n\n"
        f"RESPONSIBLE ({RESPONSIBLE_INSTRUCTIONS})\n{_fmt(RESPONSIBLE_CRITERIA)}\n\n"
        f"ESCALATE ({ESCALATE_INSTRUCTIONS}) -> YES or NO\n\n"
        f"ACTION ({ACTION_INSTRUCTIONS})\n{_fmt(ACTION_CRITERIA)}\n\n"
        'Reply with ONLY a JSON object, no other text, exactly in this form:\n'
        '{"severity": "...", "responsible": "...", "escalate": "YES|NO", "action": "..."}'
    )
    return [{"role": "system", "content": system},
            {"role": "user", "content": f"Incident: {inc.to_text()}"}]


JSON_SCHEMA = {
    "type": "object",
    "properties": {k: {"type": "string", "enum": ALLOWED[k]} for k in FIELDS},
    "required": list(FIELDS),
    "additionalProperties": False,
}


def parse_llm_text(text: str):
    """Returns (validated decision | None, strict_json_flag)."""
    text = (text or "").strip()
    try:
        return validate(json.loads(text)), True          # pure JSON -> strict success
    except Exception:
        pass
    i, j = text.find("{"), text.rfind("}")                # lenient: dig a JSON object out of chatter/fences
    if i != -1 and j > i:
        try:
            return validate(json.loads(text[i:j + 1])), False
        except Exception:
            pass
    return None, False


# =========================================================================== #
# Sonnet / Gemini through OpenRouter (OpenAI-compatible chat completions)
# =========================================================================== #
class OpenRouterBackend:
    def __init__(self, model: str, name: str, constrained: bool = False):
        self.name = name
        self.model = model
        self.provider = "openrouter"
        base = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
        self.url = base.rstrip("/") + "/chat/completions"
        self.token = os.environ.get("OPENROUTER_API_KEY")
        self.constrained = constrained
        if not self.token:
            raise RuntimeError("OPENROUTER_API_KEY is not set")

    async def decide(self, client: httpx.AsyncClient, inc: Incident) -> CallResult:
        # reasoning-style models can burn the token budget on hidden thinking -> give them room
        payload = {"model": self.model, "messages": build_llm_messages(inc),
                   "temperature": 0.0, "max_tokens": int(os.getenv("LLM_MAX_TOKENS", "800"))}
        effort = os.getenv("LLM_REASONING_EFFORT")          # optional: minimal | low | ... (OpenRouter unified param)
        if effort:
            payload["reasoning"] = {"effort": effort}
        if self.constrained:
            payload["response_format"] = {"type": "json_schema",
                                          "json_schema": {"name": "decision", "schema": JSON_SCHEMA, "strict": True}}
        headers = {"Authorization": f"Bearer {self.token}", "Content-Type": "application/json"}
        r, retries, lat, err = await post_with_retry(client, self.url, headers, payload)
        res = CallResult(latency_s=lat, retries=retries)
        if r is None:
            res.error = err
            return res
        if r.status_code != 200:
            res.error = f"HTTP {r.status_code}: {r.text[:200]}"
            res.raw = r.text[:600]
            return res
        res.transport_ok = True
        try:
            data = r.json()
            content = data["choices"][0]["message"]["content"]
            res.raw = (content or "")[:600]
            res.decision, res.strict_json = parse_llm_text(content)
            res.valid = res.decision is not None
            usage = data.get("usage") or {}
            res.input_tokens = usage.get("prompt_tokens", 0)
            res.output_tokens = usage.get("completion_tokens", 0)
            if not res.valid:
                res.error = "invalid structured output"
        except Exception as e:
            res.error = f"parse error: {type(e).__name__}: {e}"
        return res

### 4b. OTel-2.0-Decision backend (AT&T)
Reads the decision from answer-label **log-probabilities** (one 1-token request per question). The check cell below asks OpenRouter's public catalog whether the model is listed there.

In [ ]:
# =========================================================================== #
# OTel-2.0-Decision (AT&T) : typed decisions read from label log-probabilities
# =========================================================================== #
import math

class OTelBackend:
    """att/OTel-2.0-Decision-26B-A4B-4bit  (Gemma-4 26B-A4B, 4-bit W4A16).

    This is NOT a JSON-generating LLM. Per the model card, each question is asked as a multiple-choice prompt
    (state text first, then question + lettered options), the chat endpoint is asked for ONE token with
    logprobs, and the decision is the option letter with the highest probability.
        answer_mass = sum of the probabilities of the valid letters (~1.0 for a healthy readout)
    One request per question -> 4 requests per incident, run in parallel; the incident latency is their wall time.
    Env: OTEL_BASE_URL, OTEL_MODEL, OTEL_API_KEY, OTEL_MIN_MASS, OTEL_PRICE_IN_PER_M
    """
    LETTERS = "ABCDEFGHIJ"

    def __init__(self, name: str = "otel"):
        self.name = name
        self.base = os.getenv("OTEL_BASE_URL", "https://otel-2-decision-26b-a4b-4bit.cdoresearch.ai/v1").rstrip("/")
        via_or = "openrouter.ai" in self.base
        self.model = os.getenv("OTEL_MODEL") or ("att/OTel-2.0-Decision-26B-A4B-4bit" if via_or
                                                 else "OTel-2.0-Decision-26B-A4B-4bit")
        self.url = self.base + "/chat/completions"
        self.key = os.getenv("OTEL_API_KEY") or (os.environ.get("OPENROUTER_API_KEY") if via_or else "not-needed")
        self.min_mass = float(os.getenv("OTEL_MIN_MASS", "0.5"))
        if not self.key:
            raise RuntimeError("OPENROUTER_API_KEY is not set (OTEL_BASE_URL points at OpenRouter)")
        self.questions = self.build_questions()

    @staticmethod
    def build_questions() -> dict:
        """field -> (instructions, [(label, description), ...]). Same policy text the other systems get."""
        return {
            "severity": (SEVERITY_INSTRUCTIONS, list(SEVERITY_CRITERIA.items())),
            "responsible": (RESPONSIBLE_INSTRUCTIONS, list(RESPONSIBLE_CRITERIA.items())),
            "escalate": (ESCALATE_INSTRUCTIONS, [("YES", ESCALATE_TRUE), ("NO", ESCALATE_FALSE)]),
            "action": (ACTION_INSTRUCTIONS, list(ACTION_CRITERIA.items())),
        }

    def build_prompt(self, state: str, instructions: str, options: list) -> str:
        # text FIRST and byte-identical across the 4 questions, so vLLM prefix caching can reuse it
        lines = [f"State:\n{state}", "", f"Question: {instructions}", "Options:"]
        for letter, (lab, desc) in zip(self.LETTERS, options):
            lines.append(f"{letter}. {lab}: {desc}")
        lines.append("Answer with the letter only.")
        return "\n".join(lines)

    async def _ask(self, client, state: str, field_: str):
        instructions, options = self.questions[field_]
        letters = list(self.LETTERS[:len(options)])
        payload = {"model": self.model, "temperature": 0, "max_tokens": 1, "logprobs": True, "top_logprobs": 20,
                   "messages": [{"role": "user", "content": self.build_prompt(state, instructions, options)}]}
        headers = {"Authorization": f"Bearer {self.key}", "Content-Type": "application/json"}
        r, retries, lat, err = await post_with_retry(client, self.url, headers, payload)
        out = {"field": field_, "retries": retries, "latency": lat, "ok": False, "choice": None, "conf": None,
               "mass": 0.0, "probs": {}, "in_tok": 0, "out_tok": 0, "error": err}
        if r is None:
            return out
        if r.status_code != 200:
            out["error"] = f"HTTP {r.status_code}: {r.text[:200]}"
            return out
        out["ok"] = True
        try:
            data = r.json()
            tops = data["choices"][0]["logprobs"]["content"][0]["top_logprobs"]
            probs = {l: 0.0 for l in letters}
            for t in tops:
                tok = (t.get("token") or "").strip().upper()
                if tok in probs:
                    probs[tok] += math.exp(t["logprob"])
            mass = sum(probs.values())
            usage = data.get("usage") or {}
            out["in_tok"] = usage.get("prompt_tokens", 0)
            out["out_tok"] = usage.get("completion_tokens", 0)
            out["mass"] = mass
            if mass < self.min_mass:
                out["error"] = f"{field_}: low answer_mass {mass:.3f}"
                return out
            best = max(probs, key=probs.get)
            out["choice"] = options[letters.index(best)][0]
            out["conf"] = probs[best] / mass
            out["probs"] = {options[letters.index(l)][0]: round(p, 4) for l, p in probs.items()}
        except Exception as e:      # e.g. endpoint ignored logprobs -> no logprobs block in the response
            out["error"] = f"{field_}: parse error {type(e).__name__}: {e} (does the server return logprobs?)"
        return out

    async def decide(self, client, inc: Incident) -> CallResult:
        state = inc.to_text()
        t0 = time.perf_counter()
        parts = await asyncio.gather(*(self._ask(client, state, f) for f in FIELDS))
        res = CallResult(latency_s=time.perf_counter() - t0)
        res.retries = sum(p["retries"] for p in parts)
        res.transport_ok = all(p["ok"] for p in parts)
        res.input_tokens = sum(p["in_tok"] for p in parts)
        res.output_tokens = sum(p["out_tok"] for p in parts)
        res.confidence = {p["field"]: p["conf"] for p in parts}
        res.decision = validate_partial({p["field"]: p["choice"] for p in parts})
        res.valid = all(v is not None for v in res.decision.values())
        res.strict_json = True if res.transport_ok else None     # typed by construction
        errs = [p["error"] for p in parts if p["error"]]
        res.error = "; ".join(errs) if errs else None
        res.raw = json.dumps({p["field"]: {"probs": p["probs"], "mass": round(p["mass"], 3)} for p in parts})[:600]
        return res

In [ ]:
# Is the model on OpenRouter? Ask OpenRouter's public catalog directly (needs no key).
async def check_openrouter_for_otel():
    async with httpx.AsyncClient(timeout=30) as c:
        r = await c.get("https://openrouter.ai/api/v1/models")
    ids = [m["id"] for m in r.json()["data"]]
    hits = [i for i in ids if "otel" in i.lower() or i.lower().startswith("att/")]
    print(f"{len(ids)} models in the OpenRouter catalog.")
    print("OTel / att matches:", hits if hits else "none -> not on OpenRouter, so we call the AT&T endpoint (or your own vLLM server)")

await check_openrouter_for_otel()

## 5. Benchmark engine: runner, metrics, tables, plots

In [ ]:
"""
benchmark.py  --  Telecom Decision Benchmark: Jev vs Laya vs OTel vs Claude Sonnet vs Gemini

One runner (`run_benchmark`) serves the smoke test, the standard run and the combined (standard + narrative
edge-case) run, so all three are measured in exactly the same way.
"""

import csv
import json
import os
import time
from datetime import datetime

import httpx

LABELS = {"severity": list(SEVERITY_CRITERIA), "responsible": list(RESPONSIBLE_CRITERIA),
          "escalate": ["YES", "NO"], "action": list(ACTION_CRITERIA)}

LIVE_PRICES: dict = {}      # backend name -> (usd_in_per_M, usd_out_per_M), filled from OpenRouter's catalog
LAST_RUN_DIR: str | None = None   # set by run_benchmark(); the analysis cells read it


# --------------------------------------------------------------------------- #
# helpers
# --------------------------------------------------------------------------- #
async def load_live_prices(client, backends):
    """Ask OpenRouter for each LLM's real price (and warn if a slug does not exist)."""
    wanted = [b for b in backends if isinstance(b, OpenRouterBackend)]
    if not wanted:
        return
    try:
        r = await client.get("https://openrouter.ai/api/v1/models", timeout=30)
        catalog = {m["id"]: m.get("pricing", {}) for m in r.json()["data"]}
    except Exception as e:
        print(f"[prices] could not fetch OpenRouter catalog ({type(e).__name__}); costs for LLMs will show $0")
        return
    for b in wanted:
        p = catalog.get(b.model)
        if p is None:
            print(f"[prices] WARNING: '{b.model}' is not in OpenRouter's catalog -> check the slug at openrouter.ai/models")
            continue
        try:
            LIVE_PRICES[b.name] = (float(p["prompt"]) * 1e6, float(p["completion"]) * 1e6)
            print(f"[prices] {b.name} ({b.model}): ${LIVE_PRICES[b.name][0]:.3f} in / ${LIVE_PRICES[b.name][1]:.3f} out per 1M tokens")
        except (KeyError, TypeError, ValueError):
            print(f"[prices] no usable price for '{b.model}'; its cost will show $0")


def make_backend(name: str, args):
    if name == "jev":
        return JevBackend(model=args.jev_model, name="jev")
    if name == "laya":
        return LayaBackend(name="laya")
    if name == "otel":
        return OTelBackend(name="otel")
    if name == "sonnet":
        return OpenRouterBackend(model=args.sonnet_slug, name="sonnet", constrained=args.constrained)
    if name == "gemini":
        return OpenRouterBackend(model=args.gemini_slug, name="gemini", constrained=args.constrained)
    raise ValueError(f"unknown backend: {name!r} (choose from: jev, laya, otel, sonnet, gemini)")


def prices_for(name: str) -> tuple[float, float]:
    """USD per 1M (input, output) tokens."""
    if name in LIVE_PRICES:
        return LIVE_PRICES[name]
    if name == "jev":
        return float(os.getenv("JEV_PRICE_IN_PER_M", "0.042")), float(os.getenv("JEV_PRICE_OUT_PER_M", "0"))
    if name == "otel":
        return float(os.getenv("OTEL_PRICE_IN_PER_M", "0")), 0.0
    return 0.0, 0.0


def total_cost_usd(name: str, results) -> float:
    if name == "laya":      # local model: pay for compute time, not tokens
        return sum(r.compute_s for r in results) * float(os.getenv("LAYA_HOURLY_USD", "0.35")) / 3600
    pin, pout = prices_for(name)
    return sum(r.input_tokens for r in results) * pin / 1e6 + sum(r.output_tokens for r in results) * pout / 1e6


def percentile(values: list[float], p: float) -> float:
    if not values:
        return float("nan")
    s = sorted(values)
    k = (len(s) - 1) * p / 100
    lo, hi = int(k), min(int(k) + 1, len(s) - 1)
    return s[lo] + (s[hi] - s[lo]) * (k - lo)


def macro_f1(y_true: list, y_pred: list, labels: list) -> float:
    f1s = []
    for l in labels:
        tp = sum(t == l and p == l for t, p in zip(y_true, y_pred))
        fp = sum(t != l and p == l for t, p in zip(y_true, y_pred))
        fn = sum(t == l and p != l for t, p in zip(y_true, y_pred))
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return sum(f1s) / len(f1s)


# --------------------------------------------------------------------------- #
# running one (backend, concurrency) cell
# --------------------------------------------------------------------------- #
async def run_level(backend, client, incidents, concurrency: int):
    sem = asyncio.Semaphore(concurrency)
    results: list[CallResult] = [None] * len(incidents)  # type: ignore

    async def worker(i, inc):
        async with sem:
            try:
                results[i] = await backend.decide(client, inc)
            except Exception as e:                      # a backend bug must never kill the whole run
                results[i] = CallResult(error=f"unhandled {type(e).__name__}: {e}")

    t0 = time.perf_counter()
    await asyncio.gather(*(worker(i, inc) for i, inc in enumerate(incidents)))
    return results, time.perf_counter() - t0


def summarize(backend_name, concurrency, truths, results, wall) -> dict:
    n = len(results)
    ok = [r for r in results if r.transport_ok]
    valid = [r for r in results if r.valid]
    lat_ms = [r.latency_s * 1000 for r in ok]

    row = {
        "backend": backend_name, "concurrency": concurrency, "n": n,
        # --- performance
        "wall_s": round(wall, 2),
        "throughput_per_s": round(len(ok) / wall, 2) if wall else 0,
        "mean_ms": round(sum(lat_ms) / len(lat_ms), 1) if lat_ms else float("nan"),
        "p50_ms": round(percentile(lat_ms, 50), 1), "p95_ms": round(percentile(lat_ms, 95), 1),
        "p99_ms": round(percentile(lat_ms, 99), 1),
        # --- reliability
        "transport_error_rate": round(1 - len(ok) / n, 4),
        "valid_output_rate": round(len(valid) / n, 4),
        "strict_json_rate": round(sum(1 for r in ok if r.strict_json) / n, 4),
        "retries_total": sum(r.retries for r in results),
    }
    # --- quality (a failed / invalid call counts as WRONG, never silently dropped)
    for f in FIELDS:
        preds = [r.decision[f] if r.decision else None for r in results]
        gold = [t[f] for t in truths]
        row[f"acc_{f}"] = round(sum(p == g for p, g in zip(preds, gold)) / n, 4)
        if f == "severity":
            row["macro_f1_severity"] = round(macro_f1(gold, preds, LABELS[f]), 4)
    exact = sum(bool(r.decision) and all(r.decision[f] == t[f] for f in FIELDS) for r, t in zip(results, truths))
    row["exact_match"] = round(exact / n, 4)
    # --- cost
    cost = total_cost_usd(backend_name, results)
    row["avg_in_tokens"] = round(sum(r.input_tokens for r in results) / n, 1)
    row["avg_out_tokens"] = round(sum(r.output_tokens for r in results) / n, 1)
    row["cost_per_1k_usd"] = round(cost / n * 1000, 5)
    # --- Jev / Laya: does confidence mean anything?
    confs = [(r.confidence.get("severity"), r.decision["severity"] == t["severity"])
             for r, t in zip(results, truths) if r.decision and r.confidence.get("severity") is not None]
    if confs:
        hi = [c for c in confs if c[0] >= 0.9]
        lo = [c for c in confs if c[0] < 0.9]
        row["sev_acc_when_conf>=0.9"] = round(sum(c[1] for c in hi) / len(hi), 4) if hi else None
        row["sev_acc_when_conf<0.9"] = round(sum(c[1] for c in lo) / len(lo), 4) if lo else None
    return row


# --------------------------------------------------------------------------- #
# output
# --------------------------------------------------------------------------- #
def print_table(rows: list[dict]):
    bw = max([12] + [len(str(r["backend"])) for r in rows])
    cols = [("backend", bw), ("concurrency", 5), ("p50_ms", 8), ("p95_ms", 8), ("throughput_per_s", 8),
            ("exact_match", 7), ("acc_severity", 7), ("macro_f1_severity", 7), ("valid_output_rate", 7),
            ("strict_json_rate", 7), ("transport_error_rate", 7), ("cost_per_1k_usd", 10)]
    short = {"concurrency": "conc", "throughput_per_s": "req/s", "exact_match": "exact", "acc_severity": "accSev",
             "macro_f1_severity": "f1Sev", "valid_output_rate": "valid", "strict_json_rate": "strict",
             "transport_error_rate": "err", "cost_per_1k_usd": "$/1k"}
    print("\n" + " ".join(short.get(c, c).rjust(w) for c, w in cols))
    for r in rows:
        print(" ".join(str(r.get(c, "")).rjust(w) for c, w in cols))
    print()


def make_plots(rows: list[dict], outdir: str):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib not installed -> skipping plots")
        return
    names = sorted({r["backend"] for r in rows})
    levels = sorted({r["concurrency"] for r in rows})
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
    for n in names:
        rs = sorted([r for r in rows if r["backend"] == n], key=lambda r: r["concurrency"])
        xs = [r["concurrency"] for r in rs]
        line, = ax[0].plot(xs, [r["p50_ms"] for r in rs], marker="o", label=f"{n} p50")
        ax[0].plot(xs, [r["p95_ms"] for r in rs], marker="x", ls="--", color=line.get_color(), label=f"{n} p95")
        ax[1].plot(xs, [r["throughput_per_s"] for r in rs], marker="o", label=n)
    ax[0].set(title="Latency vs concurrency", xlabel="concurrent requests", ylabel="ms"); ax[0].legend(fontsize=7)
    ax[1].set(title="Throughput vs concurrency", xlabel="concurrent requests", ylabel="decisions / s"); ax[1].legend()
    base = [r for r in rows if r["concurrency"] == levels[0]]
    ax[2].bar([r["backend"] for r in base], [r["exact_match"] for r in base])
    ax[2].set(title=f"Exact-match accuracy (concurrency={levels[0]})", ylim=(0, 1))
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "benchmark.png"), dpi=140)
    plt.show()
    print(f"saved plot -> {os.path.join(outdir, 'benchmark.png')}")


# --------------------------------------------------------------------------- #
# main runner (smoke test, standard run and combined run all go through here)
# --------------------------------------------------------------------------- #
async def run_benchmark(args, incidents, run_name: str | None = None):
    """Runs every backend in args.backends over `incidents` at every level in args.concurrency.
    args.smoke=True -> one call per backend, raw output printed, nothing saved."""
    global LAST_RUN_DIR
    truths = [label(i) for i in incidents]
    backends = [make_backend(b, args) for b in args.backends]
    limits = httpx.Limits(max_connections=max(args.concurrency) + 10,
                          max_keepalive_connections=max(args.concurrency) + 10)

    async with httpx.AsyncClient(timeout=args.timeout, limits=limits) as client:
        await load_live_prices(client, backends)

        if args.smoke:
            inc = incidents[0]
            print("INCIDENT :", inc.to_text())
            print("TRUTH    :", truths[0], "\n")
            for b in backends:
                r = await b.decide(client, inc)
                print(f"[{b.name}] {r.latency_s*1000:.0f} ms | transport_ok={r.transport_ok} valid={r.valid} "
                      f"strict_json={r.strict_json}")
                print("  decision  :", r.decision)
                print("  confidence:", r.confidence or "-")
                print("  error     :", r.error)
                print("  raw       :", r.raw, "\n")
            return None, None

        outdir = os.path.join(args.out, run_name or datetime.now().strftime("%Y%m%d_%H%M%S"))
        os.makedirs(outdir, exist_ok=True)
        summary, detail = [], []

        for b in backends:
            print(f"\n== warming up {b.name} ({args.warmup} calls)")
            for inc in incidents[:args.warmup]:
                await b.decide(client, inc)
            for c in args.concurrency:
                print(f"== {b.name}: {len(incidents)} incidents @ concurrency {c}")
                results, wall = await run_level(b, client, incidents, c)
                summary.append(summarize(b.name, c, truths, results, wall))
                for inc, t, r in zip(incidents, truths, results):
                    detail.append({"backend": b.name, "concurrency": c, "incident_id": inc.incident_id,
                                   "incident_type": "complex" if isinstance(inc, ComplexIncident) else "standard",
                                   **{f"true_{f}": t[f] for f in FIELDS},
                                   **{f"pred_{f}": (r.decision or {}).get(f) for f in FIELDS},
                                   "latency_ms": round(r.latency_s * 1000, 1), "valid": r.valid,
                                   "strict_json": r.strict_json, "retries": r.retries,
                                   "error": r.error, "raw": r.raw.replace("\n", " ")[:300]})
                print_table(summary[-1:])

        print_table(summary)
        with open(os.path.join(outdir, "summary.json"), "w") as f:
            json.dump(summary, f, indent=2)
        for name, rows in (("summary.csv", summary), ("details.csv", detail)):
            with open(os.path.join(outdir, name), "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r}))
                w.writeheader(); w.writerows(rows)
        make_plots(summary, outdir)
        LAST_RUN_DIR = outdir
        print(f"results saved in {outdir}")
        return summary, detail


async def amain(args):
    """Standard run: args.n synthetic incidents (seeded, with boundary-value cases)."""
    incidents = generate_incidents(args.n, seed=args.seed, hard_fraction=args.hard_fraction)
    return await run_benchmark(args, incidents)


async def run_combined_benchmark(args, custom_incidents):
    """Combined run: standard incidents + narrative edge cases, saved in results/combined_run_results."""
    return await run_benchmark(args, custom_incidents, run_name="combined_run_results")

## 6. Smoke test (run this first)
One call per backend with the raw output printed. Confirms your keys work, Laya loads (first call downloads the checkpoint), the OTel endpoint returns logprobs, and every response shape parses, especially the `noul` (escalate) answer.

In [ ]:
from types import SimpleNamespace

cfg = SimpleNamespace(
    backends=BACKENDS,
    jev_model=JEV_MODEL, sonnet_slug=SONNET_SLUG, gemini_slug=GEMINI_SLUG,
    n=100,                           # incidents per (backend, concurrency) cell
    concurrency=[1, 5, 15],  # try [1,5,10,25,50,100] for a bigger load test
    seed=42, hard_fraction=0.3,      # share of boundary-value incidents
    warmup=3, timeout=60.0,
    constrained=False,               # True = ask the LLMs for schema-constrained JSON (hides format failures)
    out="results", smoke=True,
)
await amain(cfg)   # Jupyter supports top-level await

## 7. Full benchmark
Every backend runs the same incidents at each concurrency level. Cost: `n × backends × levels` calls, so start small (e.g. `n=50`) to check spend and rate limits.
**7a** = standard synthetic incidents. **7b** = the same standard incidents plus 3 narrative edge cases.
Laya is serialised on one model instance, so its latency rises with concurrency (queueing). That is a real property of local serving, not an error.

In [ ]:
# 7a. Standard run
cfg.smoke = False
cfg.n = 100
cfg.concurrency = [1, 5, 15]
await amain(cfg)

In [ ]:
# 7b. Combined run: standard boundary-focused incidents + 3 narrative edge cases
benchmark_incidents = generate_combined_benchmark_dataset(50, seed=42)

cfg.smoke = False
cfg.concurrency = [1, 5, 15]       # lower concurrency to stay inside OpenRouter rate limits
await run_combined_benchmark(cfg, benchmark_incidents)

## 8. Analysis of saved results
All analysis cells read `LAST_RUN_DIR` (the run you just did). To analyse an older run, set it by hand, e.g. `LAST_RUN_DIR = "results/combined_run_results"`.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
sns.set_context("paper", font_scale=1.3, rc={"lines.linewidth": 2.2, "grid.linewidth": 0.8})

if not LAST_RUN_DIR:                                   # fall back to the newest folder if nothing ran in this session
    candidates = sorted(glob.glob("results/*"), key=os.path.getmtime)
    LAST_RUN_DIR = candidates[-1] if candidates else None
assert LAST_RUN_DIR, "No results found. Run section 7 first."
print("run folder:", LAST_RUN_DIR)

summary = pd.read_csv(f"{LAST_RUN_DIR}/summary.csv")
details = pd.read_csv(f"{LAST_RUN_DIR}/details.csv")

cols = ["backend","concurrency","p50_ms","p95_ms","p99_ms","throughput_per_s","exact_match",
        "acc_severity","macro_f1_severity","acc_responsible","acc_escalate","acc_action",
        "valid_output_rate","strict_json_rate","transport_error_rate","avg_in_tokens","avg_out_tokens","cost_per_1k_usd"]
summary[[c for c in cols if c in summary.columns]].sort_values(["backend","concurrency"])

In [ ]:
# Shared colours/markers so every figure uses the same colour for the same system
backends = list(summary["backend"].unique())
palette = sns.color_palette("tab10", len(backends))
markers = ["o", "s", "^", "D", "v", "p", "*"]
color_map  = dict(zip(backends, palette))
marker_map = dict(zip(backends, markers))
plain_axis = lambda ax: (ax.set_xscale("log", base=2), ax.set_xticks(sorted(summary["concurrency"].unique())),
                         ax.get_xaxis().set_major_formatter(plt.ScalarFormatter()))

fig, axes = plt.subplots(1, 3, figsize=(20, 6), dpi=150)

# A. latency profile
ax = axes[0]
for b in backends:
    sub = summary[summary["backend"] == b].sort_values("concurrency")
    ax.plot(sub["concurrency"], sub["p50_ms"], label=f"{b} (P50)", color=color_map[b], marker=marker_map[b], markersize=8)
    ax.plot(sub["concurrency"], sub["p95_ms"], label=f"{b} (P95)", color=color_map[b], marker=marker_map[b], markersize=6, ls="--", alpha=0.7)
ax.set_title("A. Response Latency Profile (P50 vs. P95)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Concurrency (Simultaneous Requests)"); ax.set_ylabel("Response Latency (ms)")
plain_axis(ax); ax.legend(fontsize=8, loc="upper left", framealpha=0.9)

# B. throughput
ax = axes[1]
for b in backends:
    sub = summary[summary["backend"] == b].sort_values("concurrency")
    ax.plot(sub["concurrency"], sub["throughput_per_s"], label=b, color=color_map[b], marker=marker_map[b], markersize=8)
ax.set_title("B. System Throughput Scaling", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Concurrency (Simultaneous Requests)"); ax.set_ylabel("Throughput (Decisions / Second)")
plain_axis(ax); ax.legend(fontsize=9, loc="upper left")

# C. exact-match accuracy at the lowest concurrency
ax = axes[2]
acc = summary[summary["concurrency"] == summary["concurrency"].min()].sort_values("exact_match", ascending=False)
bars = ax.bar(acc["backend"], acc["exact_match"], color=[color_map[b] for b in acc["backend"]], edgecolor="black", width=0.55)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{bar.get_height():.2f}", ha="center", fontsize=10, fontweight="bold")
ax.set_title("C. Policy Quality (Exact Match, all 4 fields)", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Exact Match Rate [0-1]"); ax.set_ylim(0, 1.1)
ax.set_xticks(range(len(acc))); ax.set_xticklabels(acc["backend"], rotation=0, fontsize=11)

plt.suptitle("Comparative Performance Analysis: " + " vs. ".join(backends), fontsize=16, fontweight="bold", y=1.03)
plt.tight_layout()
p = os.path.join(LAST_RUN_DIR, "publication_quality_benchmark.png")
plt.savefig(p, dpi=300, bbox_inches="tight"); plt.show()
print("saved:", p)

In [ ]:
# Cost vs accuracy, format robustness, latency jitter (lowest concurrency)
c1 = summary[summary["concurrency"] == summary["concurrency"].min()].copy()
fig, axes = plt.subplots(1, 3, figsize=(22, 6.5), dpi=150)

# A. cost-accuracy frontier (costs clipped to 1e-6 so a $0 system still shows on the log axis)
ax = axes[0]
c1["cost_plot"] = c1["cost_per_1k_usd"].clip(lower=1e-6)
for b in backends:
    sub = c1[c1["backend"] == b]
    if not sub.empty:
        ax.scatter(sub["cost_plot"], sub["exact_match"], color=color_map[b], marker=marker_map[b], s=220, label=b, edgecolor="black", zorder=5)
ax.set_xscale("log"); ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: "{:g}".format(v)))
ax.set_xlim(c1["cost_plot"].min() * 0.2, c1["cost_plot"].max() * 5)
ax.set_ylim(max(0, c1["exact_match"].min() - 0.1), min(1.05, c1["exact_match"].max() + 0.1))
ax.set_title("A. Cost vs. Accuracy Frontier", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Operational Cost (USD per 1,000 requests)"); ax.set_ylabel("Exact Match Rate [0-1]")
ax.legend(fontsize=9, loc="lower right", framealpha=0.95)

# B. strict JSON / typed-output compliance
ax = axes[1]
cd = c1.sort_values("strict_json_rate", ascending=False)
bars = ax.bar(cd["backend"], cd["strict_json_rate"], color=[color_map[b] for b in cd["backend"]], edgecolor="black", width=0.55)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{bar.get_height()*100:.1f}%", ha="center", fontsize=9, fontweight="bold")
ax.set_title("B. Strict Structured-Output Compliance", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Strict JSON / typed output rate"); ax.set_ylim(0, 1.2)
ax.set_xticks(range(len(cd))); ax.set_xticklabels(cd["backend"], fontsize=11)

# C. latency jitter
ax = axes[2]
c1["jitter"] = c1["p95_ms"] / c1["p50_ms"]
jd = c1.sort_values("jitter")
bars = ax.bar(jd["backend"], jd["jitter"], color=[color_map[b] for b in jd["backend"]], edgecolor="black", width=0.55)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{bar.get_height():.2f}x", ha="center", fontsize=9, fontweight="bold")
ax.set_title("C. Latency Jitter (P95 / P50)", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Multiplier (higher = more volatile)"); ax.set_ylim(0, jd["jitter"].max() * 1.25)
ax.set_xticks(range(len(jd))); ax.set_xticklabels(jd["backend"], fontsize=11)

plt.suptitle("Robustness: Cost-Efficiency, Output Compliance & Latency Stability", fontsize=16, fontweight="bold", y=1.03)
plt.tight_layout()
p = os.path.join(LAST_RUN_DIR, "robustness_dimensions_benchmark.png")
plt.savefig(p, dpi=300, bbox_inches="tight"); plt.show()
print("saved:", p)

In [ ]:
# Token consumption & cost scaling across concurrency (Laya tokens are an estimate; its cost is GPU-time)
fig, axes = plt.subplots(1, 3, figsize=(22, 6.5), dpi=150)
for ax, col, title, ylabel in (
        (axes[0], "avg_in_tokens",    "A. Avg Input Tokens vs Concurrency",  "Tokens per Request"),
        (axes[1], "avg_out_tokens",   "B. Avg Output Tokens vs Concurrency", "Tokens per Request"),
        (axes[2], "cost_per_1k_usd",  "C. Operational Cost vs Concurrency",  "Cost (USD per 1,000 requests)")):
    for b in backends:
        sub = summary[summary["backend"] == b].sort_values("concurrency")
        ax.plot(sub["concurrency"], sub[col].clip(lower=1e-6) if col == "cost_per_1k_usd" else sub[col],
                label=b, color=color_map[b], marker=marker_map[b], markersize=8)
    ax.set_title(title, fontsize=14, fontweight="bold", pad=12)
    ax.set_xlabel("Concurrency (Simultaneous Requests)"); ax.set_ylabel(ylabel)
    plain_axis(ax); ax.grid(True, which="both", alpha=0.4)
axes[2].set_yscale("log"); axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: "{:g}".format(v)))
axes[0].legend(fontsize=9)
plt.suptitle("Token Consumption & Cost Scaling Across Concurrency Levels", fontsize=16, fontweight="bold", y=1.03)
plt.tight_layout()
p = os.path.join(LAST_RUN_DIR, "token_cost_scaling_benchmark.png")
plt.savefig(p, dpi=300, bbox_inches="tight"); plt.show()
print("saved:", p)

In [ ]:
# Where do the mistakes happen? Per-field accuracy, standard vs narrative edge cases, confidence calibration
d1 = details[details["concurrency"] == details["concurrency"].min()].copy()
fields = ["severity", "responsible", "escalate", "action"]
for f in fields:
    d1[f"ok_{f}"] = d1[f"true_{f}"] == d1[f"pred_{f}"]
d1["exact"] = d1[[f"ok_{f}" for f in fields]].all(axis=1)

print("== Per-field accuracy (lowest concurrency) ==")
display(d1.groupby("backend")[[f"ok_{f}" for f in fields] + ["exact"]].mean().round(3))

print("\n== Exact match by incident type ==")
display(d1.pivot_table(index="backend", columns="incident_type", values="exact", aggfunc=["mean", "count"]).round(3))

print("\n== Boundary-value sanity: where do wrong severity calls land? ==")
wrong = d1[~d1["ok_severity"]]
display(wrong.groupby(["backend", "true_severity", "pred_severity"], dropna=False).size().rename("count").reset_index().sort_values(["backend", "count"], ascending=[True, False]).head(40))

print("\n== Does confidence mean anything? (Jev and Laya only; severity accuracy at conf >=0.9 vs <0.9) ==")
conf_cols = [c for c in summary.columns if c.startswith("sev_acc_when_conf")]
if conf_cols:
    display(summary[summary["concurrency"] == summary["concurrency"].min()][["backend"] + conf_cols].dropna(subset=conf_cols, how="all"))

In [ ]:
# The 3 narrative edge cases: what did each system decide? (ground truth ignores the operator notes, see note below)
cx = d1[d1["incident_type"] == "complex"]
if cx.empty:
    print("No narrative edge cases in this run (use section 7b).")
else:
    show = ["incident_id", "backend"] + [f"true_{f}" for f in fields] + [f"pred_{f}" for f in fields]
    display(cx[show].sort_values(["incident_id", "backend"]))

In [ ]:
import os
import pandas as pd

# 1. Check current per-field accuracy for 'laya' vs 'jev'
d1 = details[details['concurrency'] == details['concurrency'].min()].copy()
fields = ['severity', 'responsible', 'escalate', 'action']
for f in fields:
    d1[f'ok_{f}'] = d1[f'true_{f}'] == d1[f'pred_{f}']
d1['exact'] = d1[[f'ok_{f}' for f in fields]].all(axis=1)

print("=== Per-Field Accuracy Metrics (Current Run) ===")
display(d1.groupby('backend')[[f'ok_{f}' for f in fields] + ['exact']].mean().round(3))

# 2. Check if Laya model output is truncated or returning empty/None results
print("\n=== Sample Predictions vs Truth for Laya ===")
laya_samples = d1[d1['backend'] == 'laya'].head(5)
for _, row in laya_samples.iterrows():
    print(f"Incident {row['incident_id']} (Type: {row['incident_type']})")
    print(f"  Truth:  Sev={row['true_severity']}, Resp={row['true_responsible']}, Esc={row['true_escalate']}, Act={row['true_action']}")
    print(f"  Pred:   Sev={row['pred_severity']}, Resp={row['pred_responsible']}, Esc={row['pred_escalate']}, Act={row['pred_action']}")
    print(f"  Raw:    {row['raw'][:200]}")
    print("-" * 40)

In [ ]:
# Force re-init with explicitly set environment configurations
os.environ["LAYA_MAX_LEN"] = "1024"
os.environ["LAYA_MODEL"] = "typed-decisions"

# Re-instantiate LayaBackend with clean parameters
from laya import Router
try:
    # Unload previous router model if possible to free VRAM
    import gc, torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

laya_test = LayaBackend()
print(f"Active Laya model in backend: {laya_test.model if laya_test.model else 'default'}")
print(f"Max context length set: {laya_test.max_len}")

# Run a single quick decision on the first incident to inspect output
test_inc = benchmark_incidents[0]
import httpx
async with httpx.AsyncClient() as client:
    res = await laya_test.decide(client, test_inc)
    print("\nTest run on Incident 0:")
    print("Truth: ", label(test_inc))
    print("Pred:  ", res.decision)
    print("Raw:   ", res.raw[:400])

In [ ]:
import httpx
# Re-run a clean call with laya_test and display the complete outputs explicitly
async def run_single_check():
    test_inc = benchmark_incidents[0]
    async with httpx.AsyncClient() as client:
        res = await laya_test.decide(client, test_inc)
        print("--- Target Incident ---")
        print(test_inc.to_text())
        print("\n--- Ground Truth Labels ---")
        print(label(test_inc))
        print("\n--- Laya Predicted Decision ---")
        print(res.decision)
        print("\n--- Laya Raw Response Metadata ---")
        print(f"Valid output format: {res.valid}")
        print(f"Compute seconds: {res.compute_s:.4f}s")
        print(f"Raw JSON snippet: {res.raw[:1000]}")

await run_single_check()

In [ ]:
# Failures: anything that was not a clean, valid answer
bad = details[(~details["valid"]) | details["error"].notna()]
print(f"{len(bad)} failed/invalid calls out of {len(details)}")
if len(bad):
    display(bad.groupby(["backend", "error"], dropna=False).size().rename("count").reset_index().sort_values("count", ascending=False))

In [ ]:
# Optional: zip the results for download (works in Colab; elsewhere it just prints the path)
import shutil
shutil.make_archive("benchmark_results", "zip", "results")
try:
    from google.colab import files
    files.download("benchmark_results.zip")
except ImportError:
    print("Not in Colab: results archive is at ./benchmark_results.zip")

### How to read the results
- **Latency (p50/p95/p99)** at concurrency 1 = single-decision speed; how it grows with concurrency = scalability. p95/p99 matter more than the mean for alarm storms.
  Remember: Jev/Sonnet/Gemini include network time; Laya is local compute, and it **queues** at higher concurrency because one model instance serves one request at a time (set `LAYA_SERIALIZE=0` to test concurrent threads on your hardware, or use Laya's `predict_batch` in production).
- **Throughput** = successful decisions per second of wall-clock time.
- **exact_match / per-field accuracy**: invalid or failed calls count as wrong.
- **valid vs strict_json**: valid = parseable with legal labels; strict = the reply was *pure* JSON. Jev and Laya are typed by construction (strict = 100% whenever they respond); the gap for Sonnet/Gemini is their format-chatter risk.
- **Confidence** (`sev_acc_when_conf>=0.9` vs `<0.9`): if accuracy is clearly higher when confidence is high, a confidence-gated hybrid (small typed model first, LLM fallback on low confidence) is worth building.
- **OTel**: the decision is `argmax` over option-letter probabilities, so `strict_json` is 100% when the endpoint answers. A call is *invalid* if the letters' total probability (`answer_mass`) is below `OTEL_MIN_MASS`. Each incident costs 4 requests, so its `req/s` counts incidents, not HTTP calls. The AT&T test endpoint is shared and has no uptime guarantee, so p95/p99 there are noisy. Token cost is $0 unless you set `OTEL_PRICE_IN_PER_M`.
- **Cost**: Jev/Sonnet/Gemini are token-based. Laya is `compute seconds × LAYA_HOURLY_USD` (set it to 0 on a free Colab GPU). Laya token columns are an estimate and carry no cost.
- **Narrative edge cases (7b)**: the ground truth is rule-based and *ignores* the operator notes, so a model that sensibly reasons about the notes is scored "wrong" there. Read those 3 rows as a qualitative check, not a leaderboard.

*Ground truth is rule-based, so this measures bounded-decision rule-following, not telecom expertise.*

# **Dataset: the policy under test**

In [ ]:
import pandas as pd

# Let's organize the rules, instructions, and target choices into clear DataFrames for review

print("=== 1. SEVERITY RULES & OPTIONS ===")
severity_data = []
for level, criteria in SEVERITY_CRITERIA.items():
    severity_data.append({"Option (Class)": level, "Criteria Description": criteria})
sev_df = pd.DataFrame(severity_data)
sev_df.style.set_caption(SEVERITY_INSTRUCTIONS)
display(sev_df)

print("\n=== 2. RESPONSIBLE TEAM RULES & OPTIONS ===")
responsible_data = []
for team, criteria in RESPONSIBLE_CRITERIA.items():
    responsible_data.append({"Team Option": team, "Ownership Scope": criteria})
resp_df = pd.DataFrame(responsible_data)
resp_df.style.set_caption(RESPONSIBLE_INSTRUCTIONS)
display(resp_df)

print("\n=== 3. ESCALATION RULE ===")
esc_data = [
    {"Escalation Option": "YES", "Rule Definition": ESCALATE_TRUE},
    {"Escalation Option": "NO", "Rule Definition": ESCALATE_FALSE}
]
esc_df = pd.DataFrame(esc_data)
esc_df.style.set_caption(ESCALATE_INSTRUCTIONS)
display(esc_df)

print("\n=== 4. REMEDIATION ACTION RULES & OPTIONS ===")
action_data = []
for action, criteria in ACTION_CRITERIA.items():
    action_data.append({"Action Option": action, "Trigger Condition": criteria})
act_df = pd.DataFrame(action_data)
act_df.style.set_caption(ACTION_INSTRUCTIONS)
display(act_df)

### Download Evaluated Incident Dataset in JSON Format

In [ ]:
import json
import os
from google.colab import files

# Prepare the dataset containing both standard and complex incidents
all_incidents = []

# Re-generate standard incidents using the benchmark parameters
standard_incidents = generate_incidents(50, seed=42, hard_fraction=0.3)
for inc in standard_incidents:
    all_incidents.append({
        "incident_id": inc.incident_id,
        "incident_type": "standard",
        "element_type": inc.element_type,
        "element_id": inc.element_id,
        "region": inc.region,
        "status": inc.status,
        "users_affected": inc.users_affected,
        "packet_loss_pct": inc.packet_loss_pct,
        "latency_ms": inc.latency_ms,
        "duration_min": inc.duration_min,
        "cause": inc.cause,
        "operator_notes": "",
        "ground_truth": label(inc)
    })

# Append the complex boundary incidents
complex_incidents = generate_complex_incidents()
for inc in complex_incidents:
    all_incidents.append({
        "incident_id": inc.incident_id,
        "incident_type": "complex",
        "element_type": inc.element_type,
        "element_id": inc.element_id,
        "region": inc.region,
        "status": inc.status,
        "users_affected": inc.users_affected,
        "packet_loss_pct": inc.packet_loss_pct,
        "latency_ms": inc.latency_ms,
        "duration_min": inc.duration_min,
        "cause": inc.cause,
        "operator_notes": inc.operator_notes,
        "ground_truth": label(inc)
    })

# Save to a local JSON file
output_filename = "evaluated_incidents_dataset.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(all_incidents, f, indent=2, ensure_ascii=False)

print(f"Successfully saved {len(all_incidents)} records to {output_filename}.")

# Trigger Google Colab download
try:
    files.download(output_filename)
    print("Download prompt triggered.")
except Exception as e:
    print(f"Could not trigger browser download automatically: {e}. You can find the file in the sidebar file explorer.")